In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import fashion_mnist

from sklearn.decomposition import PCA

print("TensorFlow version:", tf.__version__)

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)

print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)

In [ ]:
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs

        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]

        epsilon = tf.random.normal(shape=(batch, dim))

        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
latent_dim = 16

encoder_inputs = layers.Input(shape=(28, 28, 1))

x = layers.Conv2D(
    32,
    kernel_size=3,
    strides=2,
    activation="relu",
    padding="same"
)(encoder_inputs)

x = layers.Conv2D(
    64,
    kernel_size=3,
    strides=2,
    activation="relu",
    padding="same"
)(x)

x = layers.Flatten()(x)

x = layers.Dense(128, activation="relu")(x)

z_mean = layers.Dense(latent_dim, name="z_mean")(x)

z_log_var = layers.Dense(
    latent_dim,
    name="z_log_var"
)(x)

z = Sampling()([z_mean, z_log_var])

encoder = Model(
    encoder_inputs,
    [z_mean, z_log_var, z],
    name="encoder"
)

encoder.summary()

In [ ]:
latent_inputs = layers.Input(shape=(latent_dim,))

x = layers.Dense(7 * 7 * 64, activation="relu")(latent_inputs)

x = layers.Reshape((7, 7, 64))(x)

x = layers.Conv2DTranspose(
    64,
    kernel_size=3,
    strides=2,
    activation="relu",
    padding="same"
)(x)

x = layers.Conv2DTranspose(
    32,
    kernel_size=3,
    strides=2,
    activation="relu",
    padding="same"
)(x)

decoder_outputs = layers.Conv2D(
    1,
    kernel_size=3,
    activation="sigmoid",
    padding="same"
)(x)

decoder = Model(
    latent_inputs,
    decoder_outputs,
    name="decoder"
)

decoder.summary()

In [ ]:
class VAE(Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)

        self.encoder = encoder
        self.decoder = decoder

        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker
        ]

    # This was missing
    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return reconstruction

    def train_step(self, data):

        with tf.GradientTape() as tape:

            # Encode
            z_mean, z_log_var, z = self.encoder(data)

            # Decode
            reconstruction = self.decoder(z)

            # Reconstruction Loss
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.keras.losses.binary_crossentropy(
                        data, reconstruction
                    ),
                    axis=(1, 2)
                )
            )

            # KL Divergence Loss
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var
                    - tf.square(z_mean)
                    - tf.exp(z_log_var),
                    axis=1
                )
            )

            # Total Loss
            total_loss = reconstruction_loss + kl_loss

        # Calculate gradients
        grads = tape.gradient(
            total_loss,
            self.trainable_weights
        )

        # Update weights
        self.optimizer.apply_gradients(
            zip(grads, self.trainable_weights)
        )

        # Update metrics
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(
            reconstruction_loss
        )
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result()
        }

In [ ]:
vae = VAE(encoder, decoder)

vae.compile(
    optimizer=tf.keras.optimizers.Adam()
)

print("VAE compiled successfully!")

In [ ]:
history = vae.fit(
    x_train,
    epochs=30,
    batch_size=64
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history["loss"], label="Total Loss")
plt.plot(
    history.history["reconstruction_loss"],
    label="Reconstruction Loss"
)
plt.plot(
    history.history["kl_loss"],
    label="KL Divergence Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VAE Training Loss")
plt.legend()
plt.grid()

plt.show()

In [ ]:
# Select 10 test images
n = 10

sample_images = x_test[:n]

# Encode the images
z_mean, z_log_var, z = encoder.predict(sample_images)

# Reconstruct the images
reconstructed_images = decoder.predict(z)

plt.figure(figsize=(15, 4))

for i in range(n):

    # Original image
    plt.subplot(2, n, i + 1)

    plt.imshow(
        sample_images[i].squeeze(),
        cmap="gray"
    )

    plt.title("Original")
    plt.axis("off")

    # Reconstructed image
    plt.subplot(2, n, i + n + 1)

    plt.imshow(
        reconstructed_images[i].squeeze(),
        cmap="gray"
    )

    plt.title("Reconstructed")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
num_samples = 10000

latent_mean, latent_log_var, latent_vectors = encoder.predict(
    x_test[:num_samples],
    batch_size=256
)

print("Latent feature shape:", latent_mean.shape)

In [ ]:
pca = PCA(n_components=2)

latent_2d = pca.fit_transform(latent_mean)

plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    latent_2d[:, 0],
    latent_2d[:, 1],
    c=y_test[:num_samples],
    cmap="tab10",
    s=5,
    alpha=0.7
)

plt.colorbar(scatter, ticks=range(10))

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.title("PCA Visualization of VAE Latent Space")

plt.show()

In [ ]:
# Find one ankle boot image
boot_index = np.where(y_test == 9)[0][0]

# Find one sneaker image
sneaker_index = np.where(y_test == 7)[0][0]

boot_image = x_test[boot_index]
sneaker_image = x_test[sneaker_index]

print("Boot index:", boot_index)
print("Sneaker index:", sneaker_index)

In [ ]:
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(boot_image.squeeze(), cmap="gray")
plt.title("Ankle Boot")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(sneaker_image.squeeze(), cmap="gray")
plt.title("Sneaker")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Add batch dimension
boot_input = np.expand_dims(boot_image, axis=0)
sneaker_input = np.expand_dims(sneaker_image, axis=0)

# Get latent mean vectors
z_boot_mean, _, _ = encoder.predict(boot_input, verbose=0)
z_sneaker_mean, _, _ = encoder.predict(sneaker_input, verbose=0)

print("Boot latent vector shape:", z_boot_mean.shape)
print("Sneaker latent vector shape:", z_sneaker_mean.shape)

In [ ]:
# Interpolation values
alphas = np.linspace(0, 1, 7)

interpolated_images = []

for alpha in alphas:

    # Latent space interpolation
    z_interpolated = (
        (1 - alpha) * z_boot_mean
        + alpha * z_sneaker_mean
    )

    # Generate image from interpolated latent vector
    generated_image = decoder.predict(
        z_interpolated,
        verbose=0
    )

    interpolated_images.append(generated_image[0])

In [ ]:
plt.figure(figsize=(18, 4))

for i, (alpha, image) in enumerate(
    zip(alphas, interpolated_images)
):
    plt.subplot(1, len(alphas), i + 1)

    plt.imshow(
        image.squeeze(),
        cmap="gray"
    )

    plt.title(f"α = {alpha:.2f}")
    plt.axis("off")

plt.suptitle(
    "Latent Space Interpolation: Ankle Boot → Sneaker",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# Number of new images to generate
num_images = 10

# Generate random latent vectors
random_latent_vectors = np.random.normal(
    size=(num_images, latent_dim)
)

# Generate new images
generated_images = decoder.predict(
    random_latent_vectors,
    verbose=0
)

# Display generated images
plt.figure(figsize=(15, 3))

for i in range(num_images):

    plt.subplot(1, num_images, i + 1)

    plt.imshow(
        generated_images[i].squeeze(),
        cmap="gray"
    )

    plt.title(f"Image {i+1}")
    plt.axis("off")

plt.suptitle(
    "New Images Generated from Random Latent Vectors",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
encoder.save("vae_encoder.keras")

decoder.save("vae_decoder.keras")

print("Encoder and Decoder saved successfully!")